# CRF Hinglish Compressor — GPU LLM-Teacher Bootstrap

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Pipeline: clone repo → embed uncommitted CRF sources → generate 5K prompts →
deterministic baseline (fast) → LLM-teacher labels on GPU → train both CRFs →
compare on held-out slice → download winner artifacts.
Set `HF_MODEL` below (default needs no access token).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first (Runtime → Change runtime type → T4 GPU)'
print('cuda:', torch.cuda.get_device_name(0))

In [ ]:
!rm -rf /content/code-mixed-gateway
!git clone --depth 1 https://github.com/tusharsaharan/code-mixed-gateway.git /content/code-mixed-gateway
!pip -q install sklearn-crfsuite scikit-learn transformers accelerate sentence-transformers
print('deps ok')

In [ ]:
%%writefile /content/code-mixed-gateway/backend/src/gateway/modules/m2_compressor/phonetic.py
"""Phonetic normalization for Romanized Hindi (Hinglish).

Solves the *spelling entropy* problem: "kar raha hai" / "kr rha h" /
"karra h" all denote the same construction but no exact string rule can
match them all. This module normalizes chaotic surface forms to canonical
forms (normalize_chars) and computes a consonant-skeleton hash
(hinglish_phash) used as a CRF feature -- never as a standalone lookup key.
"""

from __future__ import annotations

import re

_PUNCT = ",.!?:;\"'()[]{}"
_REPEAT_3PLUS = re.compile(r"(.)\1{2,}")
_MARKER = re.compile(r"^\[\[PS\d+\]\]$")
_DEVANAGARI = re.compile(r"[\u0900-\u097F]")

#: Most frequent Hinglish abbreviations observed in the wild.
ABBREVS: dict[str, str] = {
    "h": "hai",
    "hr": "har",
    "kr": "kar",
    "kro": "karo",
    "rha": "raha",
    "rhi": "rahi",
    "rhe": "rahe",
    "mt": "mat",
    "mtt": "mat",
    "nhi": "nahi",
    "ni": "nahi",
    "nai": "nahi",
    "bt": "baat",
    "bs": "bas",
    "kch": "kuch",
    "frm": "from",
    "msg": "message",
    "plz": "please",
    "pls": "please",
    "thx": "thanks",
    "hw": "how",
    "wt": "what",
    "bcz": "because",
    "bcoz": "because",
    "yr": "yaar",
    "mra": "mera",
    "mre": "mere",
    "mri": "meri",
    "fon": "phone",
    "fone": "phone",
}

#: Negation + question tokens that must NEVER be dropped (normalized forms).
#: The CRF force-keep gate checks membership after normalize_chars().
PROTECTED_TOKENS: frozenset[str] = frozenset(
    {
        # Negation -- dropping inverts meaning
        "nahi",
        "nahin",
        "mat",
        "not",
        "never",
        "n't",
        "no",
        "don't",
        "doesn't",
        "didn't",
        # Question words -- dropping changes intent
        "kya",
        "kahan",
        "kidhar",
        "kaise",
        "kaisa",
        "kab",
        "kyun",
        "kyu",
        "kaun",
        "kiska",
        "kitna",
        "kitne",
    }
)

#: Hinglish consonant-cluster mapping applied before vowel stripping.
CLUSTER_MAP: tuple[tuple[str, str], ...] = (
    ("sh", "S"),
    ("ch", "C"),
    ("th", "T"),
    ("dh", "D"),
    ("kh", "K"),
    ("gh", "G"),
    ("ph", "F"),
    ("bh", "B"),
    ("aa", "A"),
    ("ee", "I"),
    ("oo", "U"),
)

_VOWELS = frozenset("aeiou")


def _strip_punct(token: str) -> str:
    return token.strip(_PUNCT)


def is_marker(token: str) -> bool:
    """True for protected-span markers like [[PS0]]."""
    return bool(_MARKER.match(token.strip()))


def is_protected(norm_token: str) -> bool:
    """True if a *normalized* token is a protected negation/question word."""
    return norm_token in PROTECTED_TOKENS


def normalize_chars(token: str) -> str:
    """Normalize a single token's spelling chaos to a canonical form.

    Steps: lowercase -> abbreviation expansion -> repeated-char collapse
    (consonants collapse fully; long Hindi vowels aa/ee/oo capped at 2) ->
    surrounding punctuation stripped. Markers and Devanagari pass through
    (markers byte-identical, Devanagari lowercased only).
    """
    if not token:
        return token
    if is_marker(token):
        return token
    if _DEVANAGARI.search(token):
        return token.lower()
    s = _strip_punct(token.lower())
    if not s:
        return s
    if s in ABBREVS:
        return ABBREVS[s]
    # Collapse runs of 3+ identical chars to a single char ("haaaaai" ->
    # "hai", "pleaseee" -> "please"). Legitimate doubles ("arre", "http",
    # "assignment", Hindi long vowels "aa") are preserved as-is.
    s = re.sub(r"(.)\1{2,}", r"\1", s)
    # A second abbreviation pass catches cases revealed by collapsing
    # ("krr" -> "kr" -> "kar").
    if s in ABBREVS:
        return ABBREVS[s]
    return s


def hinglish_phash(token: str) -> str:
    """Consonant-skeleton phonetic hash for Romanized Hindi.

    Maps variant spellings to the same code: raha/rha/rahi -> RH*,
    nahi/nhi -> NH, kar/kr/karo -> KR, basically/bascly -> BSKL.
    One feature among many for the CRF -- collisions are expected and
    disambiguated by suffix/prefix/neighbor features.
    """
    if not token:
        return ""
    if is_marker(token):
        return "PS"
    if _DEVANAGARI.search(token):
        # Devanagari tokens hash on first char to keep them distinct.
        return "DV" + token[0]
    s = normalize_chars(token)
    if not s:
        return ""
    # Strip trailing vowel inflection noise ("karo"/"kare" share root "kar").
    stripped = re.sub(r"[aeiou]+$", "", s)
    s = stripped or s
    for src, dst in CLUSTER_MAP:
        s = s.replace(src, dst)
    s = re.sub(r"[aeiou]", "", s)
    s = re.sub(r"(.)\1+", r"\1", s)
    # Cluster replacements introduced uppercase sentinels; normalize case.
    s = s.upper()
    return s or token[0].upper()


In [ ]:
%%writefile /content/code-mixed-gateway/backend/src/gateway/modules/m2_compressor/heuristic_passes.py
"""Fast heuristic pre-pruning (Stage 3 of the CRF pipeline).

Three safe, context-free operations that run BEFORE the CRF on masked text:
greeting strip, reduplication collapse, and a tiny unconditional filler set.
Everything context-dependent (yaar/matlab/basically/toh/na/like) belongs to
the CRF, not here.
"""

from __future__ import annotations

import re

from gateway.lexicons import _lexicons as _load_lexicons
from gateway.modules.m2_compressor.phonetic import is_marker, normalize_chars
from gateway.schemas import DropRecord

#: Universally safe drops -- interjections/hesitations with no homograph risk.
UNCONDITIONAL_FILLERS: frozenset[str] = frozenset(
    {"arre", "arrey", "arey", "um", "uh", "umm", "err", "hmm"}
)

_ECHO_INITIALS = frozenset("vw")


def strip_greeting_enhanced(text: str) -> tuple[str, DropRecord | None]:
    """Strip a sentence-initial greeting, phonetically tolerant.

    Matches greeting entries against normalized token prefixes so variants
    like "namste"/"namastey"/"heyy" are caught. A message that IS only a
    greeting is returned unchanged (nothing to compress to).
    """
    lex = _load_lexicons()
    greetings: tuple[str, ...] = lex["greetings"]  # longest-first
    stripped = text.strip()
    if not stripped:
        return text, None
    lowered = stripped.lower()
    # Greeting-only message -> keep.
    if normalize_chars(lowered.strip(",.!?:; ")) in {
        normalize_chars(g) for g in greetings
    } or lowered.strip(",.!?:; ") in {g.lower() for g in greetings}:
        return stripped, None
    first_drop: DropRecord | None = None
    # Loop so stacked greetings ("hello sir,", "namaste bhai,") strip fully.
    for _ in range(3):
        matched = False
        for g in greetings:
            pat = re.compile(r"^\s*" + re.escape(g) + r"\b[\s,!.:]*", re.IGNORECASE)
            out = pat.sub("", stripped, count=1)
            if out != stripped:
                out = out.strip()
                if not out:
                    return text.strip(), None
                stripped = out
                if first_drop is None:
                    first_drop = DropRecord(token=g, category="greeting", rule="greeting_strip")
                matched = True
                break
        if not matched:
            break
    if first_drop is not None:
        return stripped, first_drop
    # Phonetic fallback: compare normalized first-token run against
    # normalized greeting entries (catches "namste," / "heyy sir,").
    tokens = stripped.split()
    norm_tokens = [normalize_chars(t) for t in tokens]
    norm_greets = {normalize_chars(g): g for g in greetings}
    # Multi-word greetings ("good morning") need a 2-token lookahead.
    for width in (2, 1):
        if len(norm_tokens) >= width:
            cand = " ".join(norm_tokens[:width]).strip(",.!?:; ")
            if cand in norm_greets:
                rest = " ".join(tokens[width:]).lstrip(",.!?:; ").strip()
                if not rest:
                    return stripped, None
                return rest, DropRecord(
                    token=" ".join(tokens[:width]),
                    category="greeting",
                    rule="greeting_strip_phonetic",
                )
    return stripped, None


def collapse_reduplication(tokens: list[str]) -> tuple[list[str], list[DropRecord]]:
    """Collapse adjacent exact reduplication (jaldi jaldi) and echo
    reduplication (chai-vai, kaam-vaam). Comparison is on normalized forms
    so "jaldi jaldi" with spelling wobble still collapses."""
    out: list[str] = []
    drops: list[DropRecord] = []
    i = 0
    while i < len(tokens):
        if i + 1 < len(tokens) and not is_marker(tokens[i]) and not is_marker(tokens[i + 1]):
            norm_i = normalize_chars(tokens[i])
            norm_j = normalize_chars(tokens[i + 1])
            if norm_i and norm_i == norm_j:
                drops.append(
                    DropRecord(token=tokens[i + 1], category="reduplication", rule="exact")
                )
                out.append(tokens[i])
                i += 2
                continue
            if (
                norm_j
                and norm_j[0] in _ECHO_INITIALS
                and len(norm_i) > 2
                and len(norm_j) > 2
                and norm_i[-2:] == norm_j[-2:]
            ):
                drops.append(
                    DropRecord(token=tokens[i + 1], category="reduplication", rule="echo")
                )
                out.append(tokens[i])
                i += 2
                continue
        out.append(tokens[i])
        i += 1
    return out, drops


def strip_unconditional_fillers(tokens: list[str]) -> tuple[list[str], list[DropRecord]]:
    """Drop only the tiny UNCONDITIONAL_FILLERS set (normalized comparison).

    Markers pass through untouched.
    """
    out: list[str] = []
    drops: list[DropRecord] = []
    for tok in tokens:
        if is_marker(tok):
            out.append(tok)
            continue
        if normalize_chars(tok) in UNCONDITIONAL_FILLERS:
            drops.append(DropRecord(token=tok, category="filler", rule="unconditional"))
            continue
        out.append(tok)
    return out, drops


In [ ]:
%%writefile /content/code-mixed-gateway/backend/src/gateway/modules/m2_compressor/tfidf.py
"""TF-IDF information-content scores for Hinglish tokens.

A static, pre-computed token -> [0,1] map built offline from a Hinglish
corpus (see scripts/build_tfidf.py). High score = rare/informative (keep);
low score = common/redundant (drop candidate). This is a *feature* for the
CRF, never a standalone decision maker.
"""

from __future__ import annotations

import functools
import json
from pathlib import Path

_DEFAULT_OOV = 0.5


@functools.lru_cache(maxsize=4)
def _load_map(path_str: str) -> dict[str, float]:
    try:
        data = json.loads(Path(path_str).read_text(encoding="utf-8"))
        return {str(k): float(v) for k, v in data.items()}
    except Exception:
        return {}


def load_idf(path: Path | str | None) -> dict[str, float]:
    """Load the IDF map (cached). Empty dict when path is missing/corrupt."""
    if path is None:
        return {}
    return _load_map(str(path))


def get_tfidf_score(norm_token: str, idf_map: dict[str, float] | None = None) -> float:
    """Score for an already-normalized token; 0.5 default for OOV."""
    if not norm_token:
        return _DEFAULT_OOV
    if not idf_map:
        return _DEFAULT_OOV
    v = idf_map.get(norm_token)
    return float(v) if v is not None else _DEFAULT_OOV


In [ ]:
%%writefile /content/code-mixed-gateway/backend/src/gateway/modules/m2_compressor/crf_tagger.py
"""CRF sequence tagger for Hinglish Keep/Drop decisions (Stage 4).

Feature set per token: normalized form, phonetic hash, morphology
(suffix/prefix), character-bigram cues (has_rh/has_nh), positional signals,
TF-IDF information content, and a ±2 neighbor window. Protected tokens
(negation/question) and [[PSi]] markers are force-kept in post-processing,
regardless of the model's output.
"""

from __future__ import annotations

import re
from pathlib import Path
from typing import Any

from gateway.modules.m2_compressor.phonetic import (
    hinglish_phash,
    is_marker,
    is_protected,
    normalize_chars,
)
from gateway.modules.m2_compressor.tfidf import get_tfidf_score

_DEVANAGARI = re.compile(r"[\u0900-\u097F]")
_MIN_KEEP_FRACTION = 0.30


def extract_features(
    tokens: list[str],
    i: int,
    idf_map: dict[str, float] | None = None,
) -> dict[str, Any]:
    """Feature vector for token i with a ±2 neighbor window."""
    tok = tokens[i]
    norm = normalize_chars(tok)
    phash = hinglish_phash(tok)
    tfidf = get_tfidf_score(norm, idf_map)
    n = len(tokens)

    feats: dict[str, Any] = {
        "bias": 1.0,
        "token.lower": norm,
        "token.phash": phash,
        "token.len": len(tok),
        "token.is_short": len(tok) <= 2,
        "token.is_alpha": tok.isalpha(),
        "token.is_ascii": tok.isascii(),
        "token.has_devanagari": bool(_DEVANAGARI.search(tok)),
        "token.is_marker": is_marker(tok),
        "token.is_protected": is_protected(norm),
        "token.suffix1": norm[-1:],
        "token.suffix2": norm[-2:],
        "token.suffix3": norm[-3:],
        "token.prefix1": norm[:1],
        "token.prefix2": norm[:2],
        "token.prefix3": norm[:3],
        "token.has_rh": ("rh" in norm) or phash.startswith("RH"),
        "token.has_nh": ("nh" in norm) or phash == "NH",
        "token.ends_vowel": (norm[-1:] in "aeiou") if norm else False,
        "token.is_first": i == 0,
        "token.is_last": i == n - 1,
        "token.is_second": i == 1,
        "token.is_penult": i == n - 2,
        "token.position_ratio": round(i / max(1, n - 1), 4),
        "token.tfidf_score": round(tfidf, 4),
        "token.is_high_freq": tfidf > 0.8,
        "token.is_low_freq": tfidf < 0.2,
    }

    for offset in (-2, -1, 1, 2):
        j = i + offset
        prefix = f"{offset:+d}"
        if 0 <= j < n:
            ntok = tokens[j]
            nnorm = normalize_chars(ntok)
            feats[f"{prefix}.lower"] = nnorm
            feats[f"{prefix}.phash"] = hinglish_phash(ntok)
            feats[f"{prefix}.suffix2"] = nnorm[-2:]
            feats[f"{prefix}.is_marker"] = is_marker(ntok)
            feats[f"{prefix}.is_protected"] = is_protected(nnorm)
        else:
            feats[f"{prefix}.BOS"] = j < 0
            feats[f"{prefix}.EOS"] = j >= n
    return feats


def featurize(
    tokens: list[str], idf_map: dict[str, float] | None = None
) -> list[dict[str, Any]]:
    return [extract_features(tokens, i, idf_map) for i in range(len(tokens))]


def apply_safety_gates(tokens: list[str], labels: list[str]) -> list[str]:
    """Force-keep protected tokens + markers; enforce minimum-keep floor."""
    fixed = list(labels)
    for i, tok in enumerate(tokens):
        if is_marker(tok) or is_protected(normalize_chars(tok)):
            fixed[i] = "K"
    kept = sum(1 for lab in fixed if lab == "K")
    floor = max(2, int(_MIN_KEEP_FRACTION * len(tokens)))
    if kept < floor:
        fixed = ["K"] * len(tokens)
    return fixed


def is_available() -> bool:
    """True when sklearn-crfsuite is importable (pip install -e '.[crf]')."""
    try:
        import sklearn_crfsuite  # noqa: F401

        return True
    except Exception:
        return False


class CRFTagger:
    """Thin wrapper: lazy pickle load, featurize -> predict -> safety gates."""

    def __init__(
        self,
        model_path: Path | str | None = None,
        idf_map: dict[str, float] | None = None,
    ) -> None:
        self.model_path = Path(model_path) if model_path else None
        self.idf_map = idf_map or {}
        self._model = None

    @property
    def loaded(self) -> bool:
        return self._model is not None

    def _ensure(self):
        if self._model is None:
            if not is_available():
                raise ImportError("sklearn-crfsuite not installed (pip install -e '.[crf]')")
            if self.model_path is None or not self.model_path.exists():
                raise FileNotFoundError(f"CRF model not found: {self.model_path}")
            import pickle

            with self.model_path.open("rb") as fh:
                self._model = pickle.load(fh)
        return self._model

    def predict(self, tokens: list[str]) -> list[str]:
        if not tokens:
            return []
        model = self._ensure()
        feats = featurize(tokens, self.idf_map)
        raw = model.predict_single(feats)
        labels = [str(lab) for lab in raw]
        return apply_safety_gates(tokens, labels)


In [ ]:
%%writefile /content/code-mixed-gateway/backend/src/gateway/modules/m2_compressor/linguistic.py
"""Linguistic CRF compressor: 5-stage Hinglish pipeline orchestrator.

Stage 1: safety masking (mask) -- PII/code/URLs/amounts -> [[PSi]].
Stage 2: phonetic normalization is implicit (features, never rewrites output).
Stage 3: heuristic pre-pruning (greeting strip, reduplication, tiny fillers).
Stage 4: CRF Keep/Drop tagging with force-keep safety gates.
Stage 5: reconstruction (stitch kept ORIGINAL tokens, reinject spans).

Fail-closed: any failure returns the original text unchanged.
"""

from __future__ import annotations

import re
from pathlib import Path

from gateway.modules.m2_compressor.crf_tagger import CRFTagger
from gateway.modules.m2_compressor.heuristic_passes import (
    collapse_reduplication,
    strip_greeting_enhanced,
    strip_unconditional_fillers,
)
from gateway.modules.m2_compressor.phonetic import is_marker
from gateway.modules.m2_compressor.safety_span import mask, reinject
from gateway.modules.m2_compressor.tfidf import load_idf
from gateway.schemas import CompressResult, DropRecord, SafetySpan
from gateway.tokenizer import TokenCounter

_WS = re.compile(r"\s+")


class LinguisticCompressor:
    """Synchronous CRF linguistic compressor (serve path, ~3-6ms)."""

    def __init__(
        self,
        counter: TokenCounter,
        crf_model_path: Path | str | None = None,
        tfidf_path: Path | str | None = None,
    ) -> None:
        self.counter = counter
        self.crf_model_path = Path(crf_model_path) if crf_model_path else None
        self.tfidf_path = Path(tfidf_path) if tfidf_path else None
        idf_map = load_idf(self.tfidf_path) if self.tfidf_path else {}
        self._tagger = CRFTagger(self.crf_model_path, idf_map)

    @property
    def crf_ready(self) -> bool:
        """True when a model file exists and sklearn-crfsuite is importable."""
        from gateway.modules.m2_compressor.crf_tagger import is_available

        return (
            is_available()
            and self.crf_model_path is not None
            and self.crf_model_path.exists()
        )

    def _result(
        self,
        original: str,
        compressed: str,
        spans: list[SafetySpan],
        drops: list[DropRecord],
    ) -> CompressResult:
        tok_orig = self.counter.count(original)
        tok_comp = max(1, self.counter.count(compressed))
        return CompressResult(
            original=original,
            compressed=compressed,
            spans=spans,
            drops=drops,
            token_original=tok_orig,
            token_compressed=tok_comp,
            ratio=round(tok_comp / max(1, tok_orig), 6),
            method="crf",
        )

    def compress(self, text: str) -> CompressResult:
        if not text or not text.strip():
            return self._result(text, text, [], [])
        try:
            return self._compress_inner(text)
        except Exception:
            # Fail closed: original text, honestly labelled.
            tok = max(1, self.counter.count(text))
            return CompressResult(
                original=text,
                compressed=text,
                spans=[],
                drops=[],
                token_original=tok,
                token_compressed=tok,
                ratio=1.0,
                method="crf",
            )

    def _compress_inner(self, text: str) -> CompressResult:
        masked, spans = mask(text)
        drops: list[DropRecord] = []

        # Stage 3a: greeting strip.
        stripped, g_drop = strip_greeting_enhanced(masked)
        if g_drop is not None:
            drops.append(g_drop)
        tokens = stripped.split()
        if not tokens:
            # Pre-pass consumed everything but input wasn't empty:
            # fail closed to original (e.g. greeting-only message edge).
            return self._result(text, text, spans, [])

        # Stage 3b: reduplication collapse.
        tokens, red_drops = collapse_reduplication(tokens)
        drops.extend(red_drops)

        # Stage 3c: unconditional fillers.
        tokens, fil_drops = strip_unconditional_fillers(tokens)
        drops.extend(fil_drops)
        if not tokens:
            return self._result(text, text, spans, [])

        # Stage 4: CRF tagging (skip gracefully when model unavailable).
        if self.crf_ready:
            try:
                labels = self._tagger.predict(tokens)
            except Exception:
                labels = ["K"] * len(tokens)
            crf_drops = [
                DropRecord(token=tok, category="crf", rule="keep_drop")
                for tok, lab in zip(tokens, labels, strict=True)
                if lab != "K"
            ]
            drops.extend(crf_drops)
            kept = [tok for tok, lab in zip(tokens, labels, strict=True) if lab == "K"]
            # apply_safety_gates already enforces the min-keep floor, but
            # belt-and-braces: never emit an empty compression.
            tokens = kept or tokens
        # Never drop markers that survived this far (reinject would fail).
        # (Gates already force-keep them; this is a final assertion point.)
        assert all(
            t is not None for t in tokens
        )

        # Stage 5: reconstruct + reinject.
        compressed_masked = _WS.sub(" ", " ".join(tokens)).strip()
        final, ok = reinject(compressed_masked, spans)
        if not ok:
            final = text
        # Drop records for markers must never appear (they were never dropped).
        drops = [d for d in drops if not is_marker(d.token)]
        return self._result(text, final, spans, drops)


In [ ]:
%%writefile /content/code-mixed-gateway/backend/scripts/generate_hinglish_prompts.py
"""Generate diverse Hinglish training prompts (offline, deterministic).

Usage:
    python scripts/generate_hinglish_prompts.py --n 5000 --seed 7 \\
        --out data/training/prompts.jsonl [--variants 2]

Each base prompt optionally yields spelling-variant copies via
inject_spelling_variants() to teach the CRF spelling-entropy robustness.
"""

from __future__ import annotations

import argparse
import json
import random
from pathlib import Path

# (template, domain) pairs. {x} slots are filled from small vocab lists.
TEMPLATES: list[tuple[str, str]] = [
    ("yaar {place} ka wifi bahut slow chal raha hai", "campus"),
    ("arre bhai mera {item} {problem} ho raha hai", "campus"),
    ("hello sir, mera {item} {task} hai, deadline kab hai", "campus"),
    ("namaste, hostel room {num} ka {item} kaam nahi kar raha", "campus"),
    ("suno, mera phone charge nahi ho raha, charger kharab hai", "tech"),
    ("achha toh ye error aa raha hai: {error}, kya karein", "tech"),
    ("bhai mera app baar baar crash ho raha hai, please fix karo", "tech"),
    ("matlab mera {item} update ke baad slow ho gaya hai", "tech"),
    ("mera order {num} abhi tak deliver nahi hua, status kya hai", "ecommerce"),
    ("total Rs. {num} ka payment fail ho gaya, refund kab milega", "ecommerce"),
    ("please mera {item} return karo, size galat aaya hai", "ecommerce"),
    ("kal {time} baje meeting hai, agenda mail kar do na", "general"),
    ("kya {place} ka rasta bata sakte ho, jaldi pahunchna hai", "general"),
    ("basically mujhe {item} ke liye leave application likhni hai", "general"),
    ("aur suno, {item} ka bill user@example.com par bhej do", "general"),
    # Mid-sentence fillers / politeness / auxiliaries (position diversity
    # so the CRF learns fillers are droppable anywhere, not just initially).
    ("mera wifi toh bahut slow chal raha hai yaar", "campus"),
    ("bhai suno mera {item} {problem} ho gaya hai, please help karo", "campus"),
    ("yaar tumne mera {item} dekha kya, room me nahi mil raha", "campus"),
    ("hello sir please mera {item} approve kar do, bahut urgent hai", "campus"),
    ("arre mera order wala please cancel kar do na", "ecommerce"),
    ("toh basically mujhe kal {time} baje tak {item} chahiye", "general"),
    ("matlab ye wala issue phir se aa gaya hai, please dekho", "tech"),
    ("phone toh charge ho raha hai par bahut slow hai bhai", "tech"),
    # Guard-contrast cases (distill lexicon guards into the CRF):
    # mid-sentence matlab = filler (drop) vs tech/initial matlab (keep).
    ("hostel ka wifi matlab bahut slow chal raha hai", "campus"),
    ("download matlab free version for students", "tech"),
    ("matlab ek numerical tool hai", "general"),
    # bhai vocative (drop) vs kinship/genitive (keep).
    ("bhai mera assignment submit nahi hua", "campus"),
    ("mere bhai ka phone kho gaya hai", "campus"),
    ("mera bhai kal aayega", "general"),
    # Homograph keeps: like-verb, sun-noun, you-pronoun.
    ("I like this hostel room", "general"),
    ("the sun rises in the east", "general"),
    ("can you submit the form", "campus"),
    ("you know the deadline is tomorrow", "campus"),
    # Stacked initial fillers (common in the wild: both must drop).
    ("yaar matlab mera email user@example.com par bhejo na", "general"),
    ("arre yaar mera {item} {problem} ho gaya hai", "campus"),
    ("arre bhai suno mera {item} {task} hai", "campus"),
]

SLOT_VALUES: dict[str, list[str]] = {
    "place": ["hostel", "campus", "library", "canteen", "station"],
    "item": ["assignment", "form", "phone", "laptop", "wifi", "charger", "room"],
    "problem": ["submit nahi", "slow", "kharab", "missing"],
    "task": ["bharna", "submit karna", "jama karna"],
    "error": ["index out of range", "null pointer", "timeout", "404 not found"],
    "num": ["305", "12", "2,500", "15"],
    "time": ["10", "3", "5"],
}

#: Canonical -> wild spelling variants for entropy injection.
VARIANT_MAP: dict[str, list[str]] = {
    "kar": ["kr"],
    "raha": ["rha", "raha"],
    "rahi": ["rhi"],
    "rahe": ["rhe"],
    "hai": ["h", "hai"],
    "nahi": ["nhi", "nai", "nahin"],
    "mat": ["mt"],
    "mera": ["mra"],
    "mere": ["mre"],
    "meri": ["mri"],
    "kuch": ["kch"],
    "baat": ["bt"],
    "bas": ["bs"],
    "phone": ["fone", "fon"],
    "message": ["msg"],
    "please": ["plz", "pls"],
    "because": ["bcz", "bcoz"],
    "yaar": ["yr"],
    "nahin": ["nhi", "nahi"],
}


def inject_spelling_variants(prompt: str, rng: random.Random, p: float = 0.3) -> str:
    """Randomly replace tokens with wild spellings + repeat-char emphasis."""
    out: list[str] = []
    for tok in prompt.split():
        low = tok.lower().strip(",.!?;:")
        if low in VARIANT_MAP and rng.random() < p:
            variant = rng.choice(VARIANT_MAP[low])
            # Preserve trailing punctuation of the original token.
            trail = tok[len(tok.rstrip(",.!?;:")) :] if tok.rstrip(",.!?;:") != tok else ""
            lead = tok[: len(tok) - len(tok.lstrip(",.!?;:"))] if tok.lstrip(",.!?;:") != tok else ""
            out.append(lead + variant + trail)
        elif rng.random() < 0.05 and len(tok) > 3 and tok.isalpha():
            # Emphasis elongation: "hai" -> "haai".
            out.append(tok + tok[-1])
        else:
            out.append(tok)
    return " ".join(out)


def generate(n: int, seed: int, variants: int) -> list[dict]:
    rng = random.Random(seed)
    prompts: list[dict] = []
    i = 0
    while len(prompts) < n:
        template, domain = rng.choice(TEMPLATES)
        text = template
        for slot, values in SLOT_VALUES.items():
            if "{" + slot + "}" in text:
                text = text.replace("{" + slot + "}", rng.choice(values), 1)
        prompts.append({"id": f"hing-{i:05d}", "text": text, "domain": domain})
        i += 1
        for v in range(variants):
            if len(prompts) >= n:
                break
            prompts.append(
                {
                    "id": f"hing-{i:05d}",
                    "text": inject_spelling_variants(text, rng),
                    "domain": domain,
                    "variant_of": f"hing-{i - v - 1:05d}",
                }
            )
            i += 1
    return prompts[:n]


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=5000)
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--variants", type=int, default=2)
    ap.add_argument("--out", type=str, default="data/training/prompts.jsonl")
    args = ap.parse_args()

    backend_root = Path(__file__).resolve().parents[1]
    out = Path(args.out)
    if not out.is_absolute():
        out = backend_root / out
    prompts = generate(args.n, args.seed, args.variants)
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w", encoding="utf-8") as fh:
        for p in prompts:
            fh.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"wrote {len(prompts)} prompts -> {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/code-mixed-gateway/backend/scripts/bootstrap_crf_data.py
"""LLM-bootstrap Keep/Drop labels for CRF training (offline, run once).

Pipeline per prompt: mask() -> LLM compress (local Ollama qwen3:0.6b by
default, temperature 0.0) -> reinject check -> greedy token alignment on
normalized forms -> quality gates -> JSONL with (tokens, labels).

Usage:
    python scripts/bootstrap_crf_data.py --prompts data/training/prompts.jsonl \\
        --out data/training/bootstrap_filtered.jsonl [--limit 500] [--no-llm]

--no-llm uses the deterministic lexicon prune() as the teacher (fast,
offline, good enough to validate the pipeline end-to-end before spending
LLM time). Real training should use the LLM teacher.
"""

from __future__ import annotations

import argparse
import asyncio
import json
import sys
from pathlib import Path

BACKEND_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(BACKEND_ROOT / "src"))

from gateway.lexicons import prune as lexicon_prune  # noqa: E402
from gateway.modules.m2_compressor.phonetic import (  # noqa: E402
    PROTECTED_TOKENS,
    normalize_chars,
)
from gateway.modules.m2_compressor.safety_span import mask, reinject  # noqa: E402

BOOTSTRAP_SYSTEM = (
    "You are a Hinglish text compressor. Given a Hinglish (Hindi-English mixed) "
    "message, output ONLY the compressed version by dropping words that are "
    "unnecessary -- fillers, greetings, redundant particles, auxiliary verbs, "
    "and pleasantries.\n"
    "CRITICAL RULES:\n"
    "1. NEVER drop negation words (nahi, nahin, mat, not, never)\n"
    "2. NEVER drop question words (kya, kahan, kaise, kab, kyun, kaun)\n"
    "3. NEVER drop nouns, main verbs, numbers, names, or entities\n"
    "4. NEVER add new words -- only remove existing ones\n"
    "5. NEVER reorder words -- keep the original word order\n"
    "6. Keep [[PSi]] placeholders exactly as-is\n"
    "7. Output ONLY the compressed text on a single line, nothing else\n"
    "\n"
    "Examples:\n"
    "Input: yaar basically mera hostel ka wifi bahut slow chal raha hai\n"
    "Output: mera hostel ka wifi bahut slow chal\n"
    "Input: hello sir please mera form approve kar do na\n"
    "Output: mera form approve kar do\n"
    "Input: mra wifi nhi chal rha h\n"
    "Output: mra wifi nhi chal"
)


def align_tokens(original: list[str], compressed: list[str]) -> list[str]:
    """Greedy forward alignment: K if the normalized token appears next in
    the compressed stream, else D."""
    labels: list[str] = []
    comp_norm = [normalize_chars(t) for t in compressed]
    j = 0
    for tok in original:
        norm = normalize_chars(tok)
        if j < len(comp_norm) and norm == comp_norm[j]:
            labels.append("K")
            j += 1
        else:
            labels.append("D")
    return labels


def quality_ok(orig_tokens: list[str], labels: list[str]) -> tuple[bool, str]:
    """Quality gates: negation/question survival + sane compression amount."""
    dropped = {normalize_chars(t) for t, lab in zip(orig_tokens, labels, strict=True) if lab == "D"}
    if dropped & set(PROTECTED_TOKENS):
        return False, "protected_dropped"
    if content_guard_violated(orig_tokens, labels):
        return False, "content_dropped"
    if not labels:
        return False, "empty"
    drop_ratio = sum(1 for lab in labels if lab == "D") / len(labels)
    if drop_ratio > 0.70:
        return False, "over_compression"
    if drop_ratio < 0.05:
        return False, "no_compression"
    return True, "ok"


#: English/homograph content words the lexicon deliberately never drops as
#: bare unigrams. If the LLM teacher drops one, the example is rejected --
#: the label noise would teach the CRF to eat content words.
HOMOGRAPH_GUARD: frozenset[str] = frozenset({"you", "know", "sun", "like", "log", "well"})


def content_guard_violated(orig_tokens: list[str], labels: list[str]) -> bool:
    """True if a guarded content word (or tech/initial matlab) was dropped."""
    from gateway.lexicons import _lexicons as _load_lexicons

    tech_words = frozenset(
        w.lower()
        for w in _load_lexicons()["fillers_tech_guarded"].get("matlab", {}).get("tech_context", [])
    )
    norms = [normalize_chars(t) for t in orig_tokens]
    for i, (norm, lab) in enumerate(zip(norms, labels, strict=True)):
        if lab != "D":
            continue
        if norm in HOMOGRAPH_GUARD:
            return True
        if norm == "matlab" and (i == 0 or bool(set(norms) & tech_words)):
            return True
    return False


def subset_ok(orig_tokens: list[str], comp_tokens: list[str]) -> bool:
    """Every normalized compressed token must come from the original.

    Rejects hallucinations/rewordings (e.g. small local models inventing
    synonyms) that greedy alignment would otherwise mislabel as content
    drops. Multiset comparison so duplicated tokens are accounted for.
    """
    from collections import Counter

    remaining = Counter(normalize_chars(t) for t in orig_tokens)
    for t in comp_tokens:
        norm = normalize_chars(t)
        if remaining.get(norm, 0) <= 0:
            return False
        remaining[norm] -= 1
    return True


def clean_output(raw: str) -> str:
    """First non-empty line, surrounding quotes stripped (small models chat)."""
    for line in raw.splitlines():
        line = line.strip().strip("\"'").strip()
        if line:
            # Drop common chatty prefixes.
            for prefix in ("output:", "compressed:", "compressed text:"):
                if line.lower().startswith(prefix):
                    line = line[len(prefix):].strip().strip("\"'").strip()
            return line
    return ""


async def _llm_compress(
    texts: list[str],
    model: str | None = None,
    base_url: str | None = None,
    api_key: str | None = None,
    concurrency: int = 8,
    timeout: float = 120.0,
) -> list[str | None]:
    import os

    from gateway.config import get_settings
    from gateway.llm import build_client

    settings = get_settings()
    client = build_client(
        "local",
        model or settings.local.model,
        base_url or settings.local.base_url,
        api_key if api_key else settings.local.api_key or os.environ.get("OPENAI_API_KEY", ""),
        dry_run=False,
        timeout=timeout,
    )
    sem = asyncio.Semaphore(max(1, concurrency))

    async def one(text: str) -> str | None:
        async with sem:
            try:
                res = await client.chat(
                    [
                        {"role": "system", "content": BOOTSTRAP_SYSTEM},
                        {"role": "user", "content": text},
                    ],
                    temperature=0.0,
                )
                return clean_output(res.content)
            except Exception:
                return None

    return await asyncio.gather(*[one(t) for t in texts])


def _hf_compress(
    texts: list[str],
    hf_model: str,
    batch_size: int = 8,
    max_new_tokens: int = 60,
    on_batch=None,
) -> list[str | None]:
    """Synchronous HuggingFace GPU teacher (for Colab: device_map=auto).

    Batched text-generation with the same few-shot system prompt as the API
    path. Requires transformers + accelerate + a CUDA GPU. When `on_batch`
    is given, it is called as on_batch(start_index, batch_outputs) after
    every batch so callers can flush results incrementally (crash-safe).
    """
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(hf_model)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # Left-padding is required for correct batched generation with
    # decoder-only models (right-padding lets short sequences in a batch
    # attend to trailing pad tokens, degrading their outputs).
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        hf_model,
        device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    )
    model.eval()

    out: list[str | None] = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        chats = [
            tok.apply_chat_template(
                [
                    {"role": "system", "content": BOOTSTRAP_SYSTEM},
                    {"role": "user", "content": t},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            for t in batch
        ]
        enc = tok(chats, return_tensors="pt", padding=True, truncation=True, max_length=512)
        enc = {k: v.to(model.device) for k, v in enc.items()}
        try:
            with torch.no_grad():
                gen = model.generate(
                    **enc,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tok.pad_token_id,
                    eos_token_id=tok.eos_token_id,
                )
            for i, _t in enumerate(batch):
                prompt_len = enc["input_ids"][i].shape[0]
                decoded = tok.decode(gen[i][prompt_len:], skip_special_tokens=True)
                out.append(clean_output(decoded))
        except Exception:
            out.extend([None] * len(batch))
        print(f"hf teacher: {min(start + batch_size, len(texts))}/{len(texts)}", flush=True)
        if on_batch is not None:
            on_batch(start, out[start : start + len(batch)])
    return out


def _teacher_no_llm(masked: str) -> str:
    """Deterministic teacher: lexicon prune + auxiliary-verb drop.

    The lexicon prune handles fillers/greetings/vocatives; the aux pass
    teaches what the LLM teacher would do with progressive/present
    auxiliaries (raha/rahi/rahe + hai/hain/ho...), which carry almost no
    information once the main verb survives. Comparison is on normalized
    forms so spelling variants (rha/h) are dropped too.
    """
    pruned, _ = lexicon_prune(masked)
    toks = pruned.split()
    kept: list[str] = []
    for t in toks:
        if not kept:
            kept.append(t)
            continue
        from gateway.modules.m2_compressor.phonetic import is_marker as _is_marker

        if _is_marker(t):
            kept.append(t)
            continue
        if normalize_chars(t) in AUX_TOKENS:
            continue  # drop auxiliary
        kept.append(t)
    return " ".join(kept)


#: Normalized auxiliary-verb forms (progressive + present/past be).
#: Disjoint from PROTECTED_TOKENS by construction (asserted in tests).
AUX_TOKENS: frozenset[str] = frozenset(
    {"raha", "rahi", "rahe", "hai", "hain", "hun", "tha", "thi", "the", "hoga", "hogi", "honge"}
)


def process_one(
    rec_id: str,
    masked: str,
    spans: list,
    comp: str | None,
    apply_subset_gate: bool,
) -> tuple[str | None, dict | None]:
    """Align + gate a single teacher output.

    Returns (skip_reason, row): skip_reason is None when the row is kept.
    """
    if not comp or not comp.strip():
        return "empty_response", None
    restored, ok = reinject(comp.strip(), spans)
    if not ok:
        return "marker_lost", None
    orig_tokens = masked.split()
    comp_tokens = comp.strip().split()
    if len(comp_tokens) >= len(orig_tokens):
        return "not_shorter", None
    if apply_subset_gate and not subset_ok(orig_tokens, comp_tokens):
        return "hallucinated_tokens", None
    labels = align_tokens(orig_tokens, comp_tokens)
    good, reason = quality_ok(orig_tokens, labels)
    if not good:
        return reason, None
    return None, {"id": rec_id, "tokens": orig_tokens, "labels": labels}


def load_done_ids(out_path: Path) -> set[str]:
    """Ids already present in a (possibly partial) output file."""
    if not out_path.exists():
        return set()
    done: set[str] = set()
    with out_path.open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                done.add(json.loads(line)["id"])
            except Exception:
                continue
    return done


async def main_async(args) -> None:
    prompts_path = Path(args.prompts)
    if not prompts_path.is_absolute():
        prompts_path = BACKEND_ROOT / prompts_path
    records = [
        json.loads(line)
        for line in prompts_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if args.limit:
        records = records[: args.limit]

    out_path = Path(args.out)
    if not out_path.is_absolute():
        out_path = BACKEND_ROOT / out_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if args.resume:
        done_ids = load_done_ids(out_path)
        before = len(records)
        records = [r for r in records if r.get("id") not in done_ids]
        print(f"resume: {len(done_ids)} already done, {len(records)}/{before} remaining")
        mode = "a"
    else:
        mode = "w"

    masked_texts: list[str] = []
    span_lists: list = []
    for rec in records:
        masked, spans = mask(rec["text"])
        masked_texts.append(masked)
        span_lists.append(spans)

    kept = skipped = 0
    skip_reasons: dict[str, int] = {}

    def _record(skip_reason: str | None, row: dict | None, fh) -> None:
        nonlocal kept, skipped
        if skip_reason is not None:
            skipped += 1
            skip_reasons[skip_reason] = skip_reasons.get(skip_reason, 0) + 1
            return
        assert row is not None
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")
        kept += 1

    with out_path.open(mode, encoding="utf-8") as fh:
        if args.no_llm:
            for rec, masked, spans in zip(records, masked_texts, span_lists, strict=True):
                reason, row = process_one(rec["id"], masked, spans, _teacher_no_llm(masked), False)
                _record(reason, row, fh)
        elif args.hf_model:
            def _on_batch(start: int, outs: list[str | None]) -> None:
                for offset, comp in enumerate(outs):
                    idx = start + offset
                    reason, row = process_one(
                        records[idx]["id"], masked_texts[idx], span_lists[idx], comp, True
                    )
                    _record(reason, row, fh)
                fh.flush()

            _hf_compress(masked_texts, args.hf_model, args.hf_batch, on_batch=_on_batch)
        else:
            compressed_list = await _llm_compress(
                masked_texts,
                model=args.model,
                base_url=args.base_url,
                api_key=args.api_key,
                concurrency=args.concurrency,
                timeout=args.timeout,
            )
            for rec, masked, spans, comp in zip(
                records, masked_texts, span_lists, compressed_list, strict=True
            ):
                reason, row = process_one(rec["id"], masked, spans, comp, True)
                _record(reason, row, fh)
    print(f"kept {kept} / skipped {skipped} -> {out_path}")
    if skip_reasons:
        print("skip reasons:", json.dumps(skip_reasons, indent=2))


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--prompts", default="data/training/prompts.jsonl")
    ap.add_argument("--out", default="data/training/bootstrap_filtered.jsonl")
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--no-llm", action="store_true")
    ap.add_argument("--model", default="", help="LLM model (default: settings.local.model)")
    ap.add_argument("--base-url", default="", help="API base URL (default: settings.local.base_url)")
    ap.add_argument("--api-key", default="", help="API key (default: settings key or OPENAI_API_KEY)")
    ap.add_argument("--concurrency", type=int, default=8)
    ap.add_argument("--timeout", type=float, default=120.0)
    ap.add_argument("--hf-model", default="", help="HF model id for local GPU teacher (Colab)")
    ap.add_argument("--hf-batch", type=int, default=8)
    ap.add_argument("--resume", action="store_true", help="Skip ids already in --out (crash-safe reruns)")
    args = ap.parse_args()
    asyncio.run(main_async(args))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/code-mixed-gateway/backend/scripts/build_tfidf.py
"""Build the TF-IDF information-content map (offline, run once).

Usage:
    python scripts/build_tfidf.py --prompts data/training/prompts.jsonl \\
        --out data/tfidf/hinglish_idf.json

Token keys are normalize_chars() forms; values are log-IDF normalized to
[0,1]. High = rare/informative (keep); low = common/redundant.
"""

from __future__ import annotations

import argparse
import json
import math
import sys
from collections import Counter
from pathlib import Path

BACKEND_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(BACKEND_ROOT / "src"))

from gateway.modules.m2_compressor.phonetic import normalize_chars  # noqa: E402
from gateway.modules.m2_compressor.safety_span import mask  # noqa: E402


def build(prompts: list[str]) -> dict[str, float]:
    n = len(prompts)
    df: Counter[str] = Counter()
    for text in prompts:
        masked, _ = mask(text)
        toks = {normalize_chars(t) for t in masked.split() if normalize_chars(t)}
        for tok in toks:
            df[tok] += 1
    idf = {tok: math.log(n / (1 + c)) for tok, c in df.items()}
    peak = max(idf.values()) if idf else 1.0
    return {tok: round(v / peak, 4) for tok, v in idf.items()}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--prompts", default="data/training/prompts.jsonl")
    ap.add_argument("--out", default="data/tfidf/hinglish_idf.json")
    args = ap.parse_args()

    prompts_path = Path(args.prompts)
    if not prompts_path.is_absolute():
        prompts_path = BACKEND_ROOT / prompts_path
    texts = [
        json.loads(line)["text"]
        for line in prompts_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    idf = build(texts)
    out = Path(args.out)
    if not out.is_absolute():
        out = BACKEND_ROOT / out
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(idf, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"wrote {len(idf)} entries -> {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/code-mixed-gateway/backend/scripts/train_crf.py
"""Train the CRF Hinglish compressor (offline).

Usage:
    python scripts/train_crf.py --data data/training/bootstrap_filtered.jsonl \\
        --model data/models/crf_compressor.pkl [--dev-frac 0.15] [--seed 7]

Splits data into train/dev, trains sklearn-crfsuite LBFGS, reports weighted
F1 on dev, and pickles the model.
"""

from __future__ import annotations

import argparse
import json
import pickle
import random
import sys
from pathlib import Path

BACKEND_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(BACKEND_ROOT / "src"))

from gateway.modules.m2_compressor.crf_tagger import featurize  # noqa: E402
from gateway.modules.m2_compressor.tfidf import load_idf  # noqa: E402


def load_examples(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def train(
    examples: list[dict],
    idf_map: dict[str, float],
    dev_frac: float,
    seed: float,
    model_out: Path,
) -> dict:
    import sklearn_crfsuite
    from sklearn_crfsuite import metrics

    rng = random.Random(seed)
    idx = list(range(len(examples)))
    rng.shuffle(idx)
    n_dev = max(1, int(len(examples) * dev_frac))
    dev_idx = set(idx[:n_dev])

    train_ex = [ex for k, ex in enumerate(examples) if k not in dev_idx]
    dev_ex = [ex for k, ex in enumerate(examples) if k in dev_idx]

    X_train = [featurize(ex["tokens"], idf_map) for ex in train_ex]
    y_train = [ex["labels"] for ex in train_ex]
    X_dev = [featurize(ex["tokens"], idf_map) for ex in dev_ex]
    y_dev = [ex["labels"] for ex in dev_ex]

    crf = sklearn_crfsuite.CRF(
        algorithm="lbfgs",
        c1=0.1,
        c2=0.1,
        max_iterations=200,
        all_possible_transitions=True,
    )
    crf.fit(X_train, y_train)
    y_pred = crf.predict(X_dev)
    f1 = metrics.flat_f1_score(y_dev, y_pred, average="weighted")

    model_out.parent.mkdir(parents=True, exist_ok=True)
    with model_out.open("wb") as fh:
        pickle.dump(crf, fh)

    # Top transition/feature weights for auditability.
    top_trans = sorted(crf.transition_features_.items(), key=lambda kv: -abs(kv[1]))[:10]
    top_state = sorted(crf.state_features_.items(), key=lambda kv: -abs(kv[1]))[:10]
    return {
        "f1_weighted": round(float(f1), 4),
        "train_size": len(train_ex),
        "dev_size": len(dev_ex),
        "model_path": str(model_out),
        "top_transitions": [(str(k), round(float(v), 4)) for k, v in top_trans],
        "top_state_features": [(str(k), round(float(v), 4)) for k, v in top_state],
    }


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/training/bootstrap_filtered.jsonl")
    ap.add_argument("--model", default="data/models/crf_compressor.pkl")
    ap.add_argument("--idf", default="data/tfidf/hinglish_idf.json")
    ap.add_argument("--dev-frac", type=float, default=0.15)
    ap.add_argument("--seed", type=int, default=7)
    args = ap.parse_args()

    data_path = Path(args.data)
    if not data_path.is_absolute():
        data_path = BACKEND_ROOT / data_path
    idf_path = Path(args.idf)
    if not idf_path.is_absolute():
        idf_path = BACKEND_ROOT / idf_path
    model_out = Path(args.model)
    if not model_out.is_absolute():
        model_out = BACKEND_ROOT / model_out

    examples = load_examples(data_path)
    idf_map = load_idf(idf_path) if idf_path.exists() else {}
    report = train(examples, idf_map, args.dev_frac, args.seed, model_out)
    print(json.dumps(report, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/code-mixed-gateway/backend/scripts/compare_teachers.py
"""Compare CRF compressor variants on a held-out prompt slice.

Metrics per model: mean compression ratio, mean semantic cosine
(original vs compressed, via sentence-transformers), negation-preservation
rate. Prints a JSON table; exits non-zero if any model drops a negation.

Usage:
    python scripts/compare_teachers.py --prompts data/training/prompts.jsonl \\
        --models det=data/models/crf_compressor.pkl llm=data/models/crf_llm.pkl \\
        --limit 300 --seed 999 [--no-embed]
"""

from __future__ import annotations

import argparse
import json
import random
import sys
from pathlib import Path

BACKEND_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(BACKEND_ROOT / "src"))

from gateway.modules.m2_compressor.linguistic import LinguisticCompressor  # noqa: E402
from gateway.modules.m2_compressor.phonetic import PROTECTED_TOKENS, normalize_chars  # noqa: E402
from gateway.tokenizer import TokenCounter  # noqa: E402

NEGATIONS = {t for t in PROTECTED_TOKENS if t not in {
    "kya", "kahan", "kidhar", "kaise", "kaisa", "kab",
    "kyun", "kyu", "kaun", "kiska", "kitna", "kitne",
}}


def load_texts(prompts_path: Path, limit: int, seed: int) -> list[str]:
    recs = [
        json.loads(line)
        for line in prompts_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    rng = random.Random(seed)
    rng.shuffle(recs)
    return [r["text"] for r in recs[:limit]]


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--prompts", default="data/training/prompts.jsonl")
    ap.add_argument("--models", nargs="+", required=True, help="NAME=PATH pairs")
    ap.add_argument("--idf", default="data/tfidf/hinglish_idf.json")
    ap.add_argument("--limit", type=int, default=300)
    ap.add_argument("--seed", type=int, default=999)
    ap.add_argument("--no-embed", action="store_true")
    ap.add_argument("--embed-model", default="paraphrase-multilingual-MiniLM-L12-v2")
    args = ap.parse_args()

    prompts_path = BACKEND_ROOT / args.prompts if not Path(args.prompts).is_absolute() else Path(args.prompts)
    idf_path = BACKEND_ROOT / args.idf if not Path(args.idf).is_absolute() else Path(args.idf)
    texts = load_texts(prompts_path, args.limit, args.seed)

    specs: dict[str, Path] = {}
    for item in args.models:
        name, path = item.split("=", 1)
        p = Path(path)
        specs[name] = p if p.is_absolute() else BACKEND_ROOT / p

    embedder = None
    if not args.no_embed:
        from sentence_transformers import SentenceTransformer

        embedder = SentenceTransformer("sentence-transformers/" + args.embed_model)

    table: dict[str, dict] = {}
    failed = False
    for name, model_path in specs.items():
        comp = LinguisticCompressor(TokenCounter(), crf_model_path=model_path, tfidf_path=idf_path)
        ratios: list[float] = []
        neg_kept = neg_total = 0
        pairs: list[tuple[str, str]] = []
        for text in texts:
            try:
                r = comp.compress(text)
            except Exception:
                continue
            ratios.append(r.ratio)
            pairs.append((text, r.compressed))
            # Token-based (normalized) matching: substring checks false-positive
            # on "mat" inside "matlab" or "no" inside "know".
            in_toks = {normalize_chars(t) for t in text.lower().split()}
            out_toks = {normalize_chars(t) for t in r.compressed.lower().split()}
            present = [w for w in NEGATIONS if w in in_toks]
            neg_total += len(present)
            neg_kept += sum(1 for w in present if w in out_toks)
        row: dict[str, float | str] = {
            "n": len(ratios),
            "mean_ratio": round(sum(ratios) / max(1, len(ratios)), 4),
            "negation_preserved": round(neg_kept / max(1, neg_total), 4),
        }
        if embedder is not None and pairs:
            import torch.nn.functional as F

            embs = embedder.encode([p[0] for p in pairs] + [p[1] for p in pairs], convert_to_tensor=True)
            n = len(pairs)
            sims = F.cosine_similarity(embs[:n], embs[n:]).tolist()
            row["mean_semantic_cosine"] = round(sum(sims) / len(sims), 4)
            row["min_semantic_cosine"] = round(min(sims), 4)
        if row["negation_preserved"] < 1.0:
            failed = True
        table[name] = row

    print(json.dumps(table, indent=2))
    if failed:
        print("WARNING: at least one model dropped a negation", file=sys.stderr)


if __name__ == "__main__":
    main()


In [ ]:
%cd /content/code-mixed-gateway/backend
!python scripts/generate_hinglish_prompts.py --n 5000 --seed 7 --variants 2 \
  --out data/training/prompts.jsonl
!python scripts/build_tfidf.py --prompts data/training/prompts.jsonl --out data/tfidf/hinglish_idf.json

In [ ]:
# Deterministic baseline (CPU, ~1 min)
!python scripts/bootstrap_crf_data.py --prompts data/training/prompts.jsonl --out data/training/bootstrap_det.jsonl --no-llm
!python scripts/train_crf.py --data data/training/bootstrap_det.jsonl --model data/models/crf_det.pkl --idf data/tfidf/hinglish_idf.json

In [ ]:
# LLM teacher on GPU. --resume makes reruns continue where a crash stopped
# (results flush to disk after every batch, so nothing is ever lost).
HF_MODEL = 'mistralai/Mistral-7B-Instruct-v0.1'  # or 'Qwen/Qwen2.5-7B-Instruct' (needs token + license)
HF_BATCH = 8
!python scripts/bootstrap_crf_data.py --prompts data/training/prompts.jsonl --out data/training/bootstrap_llm.jsonl --hf-model "$HF_MODEL" --hf-batch "$HF_BATCH" --resume
!python scripts/train_crf.py --data data/training/bootstrap_llm.jsonl --model data/models/crf_llm.pkl --idf data/tfidf/hinglish_idf.json

In [ ]:
# Head-to-head on a held-out slice (seed 999, never used in training)
!python scripts/compare_teachers.py --models det=data/models/crf_det.pkl llm=data/models/crf_llm.pkl --limit 500 --seed 999 | tee data/training/teacher_comparison.json

In [ ]:
import os
from google.colab import files
wanted = {
    'data/models/crf_llm.pkl': 'train cell (train_crf.py on bootstrap_llm.jsonl)',
    'data/training/bootstrap_llm.jsonl': 'HF bootstrap cell (re-run it: --resume continues)',
    'data/training/teacher_comparison.json': 'compare cell',
}
for path, rerun in wanted.items():
    if os.path.exists(path):
        files.download(path)
    else:
        print(f'MISSING {path} -> re-run the {rerun}')
print('Copy crf_llm.pkl over backend/data/models/crf_compressor.pkl locally if the llm row wins.')
print('Then re-run: pytest tests/test_linguistic_* tests/test_m2.py tests/test_lexicon.py')